In [2]:
import itertools

# ============================================================
# MDP = (S, A, P, R, gamma)
# Traffic Signal Control at a Four-Way Intersection
# ============================================================

print("Name: Someshwar S")
print("Register Number: 212224040322")
print()

# ---------------- STATE SPACE ----------------

QUEUE_LEVELS = ["Low", "Medium", "High"]
PHASES = ["NS_Green", "EW_Green"]

# Low = 0-5 vehicles
# Medium = 6-15 vehicles
# High = 16+ vehicles

def generate_states():
    states = []

    for n, s, e, w in itertools.product(QUEUE_LEVELS, repeat=4):
        for phase in PHASES:
            states.append((n, s, e, w, phase))

    return states


S = generate_states()

# Give every state a unique integer ID
STATE_ID = {state: i for i, state in enumerate(S)}
ID_STATE = {i: state for i, state in enumerate(S)}

# ---------------- ACTION SPACE ----------------

A = [
    "Keep_Current_Phase",
    "Switch_Phase"
]

ACTION_ID = {
    "Keep_Current_Phase": 0,
    "Switch_Phase": 1
}

# ---------------- REWARD PARAMETERS ----------------

QUEUE_VALUE = {
    "Low": 3,
    "Medium": 10,
    "High": 20
}

SWITCH_PENALTY = 5
GAMMA = 0.9


# ============================================================
# REWARD FUNCTION
# R(s,a,s') = -(total vehicles queued in s') - 5
#             if action is Switch_Phase
# ============================================================

def reward_function(state, action, next_state):

    n, s, e, w, phase = next_state

    total_wait = (
        QUEUE_VALUE[n]
        + QUEUE_VALUE[s]
        + QUEUE_VALUE[e]
        + QUEUE_VALUE[w]
    )

    reward = -total_wait

    if action == "Switch_Phase":
        reward -= SWITCH_PENALTY

    return reward


# ============================================================
# TRANSITION DISTRIBUTION
# ============================================================

def next_level_distribution(level, is_green):

    idx = QUEUE_LEVELS.index(level)

    if is_green:
        # 70% chance queue decreases one level
        # 30% chance queue remains unchanged
        if idx == 0:
            return {
                "Low": 1.0
            }

        return {
            QUEUE_LEVELS[idx - 1]: 0.7,
            QUEUE_LEVELS[idx]: 0.3
        }

    else:
        # 60% chance queue increases one level
        # 40% chance queue remains unchanged
        if idx == 2:
            return {
                "High": 1.0
            }

        return {
            QUEUE_LEVELS[idx + 1]: 0.6,
            QUEUE_LEVELS[idx]: 0.4
        }


# ============================================================
# BUILD P IN THE REQUIRED FORMAT
#
# P[state][action] =
# [
#     (probability, next_state, reward, done),
#     ...
# ]
# ============================================================

P = {}

for state in S:

    P[STATE_ID[state]] = {}

    for action in A:

        n, s, e, w, phase = state

        # Determine the signal phase for next step
        if action == "Switch_Phase":
            if phase == "NS_Green":
                new_phase = "EW_Green"
            else:
                new_phase = "NS_Green"
        else:
            new_phase = phase

        # Determine which directions have green
        ns_green = (new_phase == "NS_Green")

        dist_n = next_level_distribution(n, ns_green)
        dist_s = next_level_distribution(s, ns_green)

        dist_e = next_level_distribution(e, not ns_green)
        dist_w = next_level_distribution(w, not ns_green)

        transitions = []

        # Combine independent probabilities
        for nn, pn in dist_n.items():
            for ss, ps in dist_s.items():
                for ee, pe in dist_e.items():
                    for ww, pw in dist_w.items():

                        probability = pn * ps * pe * pw

                        next_state = (
                            nn,
                            ss,
                            ee,
                            ww,
                            new_phase
                        )

                        next_state_id = STATE_ID[next_state]

                        reward = reward_function(
                            state,
                            action,
                            next_state
                        )

                        # Traffic control is a continuing process,
                        # therefore no terminal states.
                        done = False

                        transitions.append(
                            (
                                round(probability, 5),
                                next_state_id,
                                reward,
                                done
                            )
                        )

        P[STATE_ID[state]][ACTION_ID[action]] = transitions


# ============================================================
# OUTPUT
# ============================================================

print("MDP Representation")
print("------------------")

print("Number of states |S| =", len(S))
print("Number of actions |A| =", len(A))
print("Discount factor gamma =", GAMMA)

print()

# Sample state
sample_state = (
    "High",
    "Low",
    "Medium",
    "Low",
    "NS_Green"
)

sample_state_id = STATE_ID[sample_state]

print("Sample state:")
print("State ID =", sample_state_id)
print("State =", sample_state)

print()

# Sample action
sample_action = "Switch_Phase"
sample_action_id = ACTION_ID[sample_action]

print("Sample action:")
print("Action ID =", sample_action_id)
print("Action =", sample_action)

print()

# Display P for the sample state/action
print("P[sample_state][sample_action] =")

for transition in P[sample_state_id][sample_action_id]:

    probability, next_state_id, reward, done = transition

    print(
        " ",
        (
            probability,
            next_state_id,
            reward,
            done
        ),
        "->",
        ID_STATE[next_state_id]
    )

Name: Someshwar S
Register Number: 212224040322

MDP Representation
------------------
Number of states |S| = 162
Number of actions |A| = 2
Discount factor gamma = 0.9

Sample state:
State ID = 114
State = ('High', 'Low', 'Medium', 'Low', 'NS_Green')

Sample action:
Action ID = 1
Action = Switch_Phase

P[sample_state][sample_action] =
  (0.42, 127, -41, False) -> ('High', 'Medium', 'Low', 'Low', 'EW_Green')
  (0.18, 133, -48, False) -> ('High', 'Medium', 'Medium', 'Low', 'EW_Green')
  (0.28, 109, -34, False) -> ('High', 'Low', 'Low', 'Low', 'EW_Green')
  (0.12, 115, -41, False) -> ('High', 'Low', 'Medium', 'Low', 'EW_Green')
